# Week 5 Lab - Clock glitching the counting loop

Make `glitch_loop` return a count that is not 2500 by glitching the **clock**, then sweep the glitch parameters and map the sensitive region.

Based on ChipWhisperer `fault101` Fault 1_1. The notebook finds your ChipWhisperer install
automatically, so it runs from anywhere.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM  = 'CW312_SAM4S'
SS_VER    = 'SS_VER_2_1'

In [ ]:
# Locate the ChipWhisperer install from the installed package, so this notebook
# runs from anywhere - it does not need to sit inside the chipwhisperer tree.
import os, chipwhisperer as cw
CWROOT = os.path.normpath(os.path.join(os.path.dirname(cw.__file__), "..", ".."))
SETUP  = os.path.join(CWROOT, "jupyter", "Setup_Scripts", "Setup_Generic.ipynb")
FW_SRC = os.path.join(CWROOT, "firmware", "mcu", "simpleserial-glitch")
# drive-less, forward-slash root for the %%bash build cell; the cell itself finds
# the right mount prefix (/mnt/c on WSL, /c on Git-Bash, / on macOS/Linux).
CW_BASH = (CWROOT[0].lower() + CWROOT[2:].replace("\\", "/")) if os.name == "nt" else CWROOT.lstrip("/")
print("ChipWhisperer:", CWROOT)

In [ ]:
%run "$SETUP"    # brings up scope, prog, target

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER" "$CW_BASH"
# find the firmware dir under whichever mount this bash uses (WSL /mnt/c, Git-Bash /c, macOS /)
FW=""
for p in "/mnt/$3" "/$3"; do
  [ -d "$p/firmware/mcu/simpleserial-glitch" ] && FW="$p/firmware/mcu/simpleserial-glitch" && break
done
[ -z "$FW" ] && { echo "firmware dir not found for root: $3" >&2; exit 1; }
cd "$FW"
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = os.path.join(FW_SRC, "simpleserial-glitch-{}.hex".format(PLATFORM))
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [ ]:
def reboot_flush():
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.nrst = "high_z"
    time.sleep(0.05)
    target.flush()

## The code we are glitching

The `'g'` command runs this on the target:

```c
uint8_t glitch_loop(uint8_t* in)
{
    volatile uint16_t i, j;
    volatile uint32_t cnt;
    cnt = 0;
    trigger_high();
    for(i=0; i<50; i++){
        for(j=0; j<50; j++){
            cnt++;
        }
    }
    trigger_low();
    simpleserial_put('r', 4, (uint8_t*)&cnt);
    return (cnt != 2500);
}
```

`volatile` stops the compiler optimising the loop into a constant. `trigger_high()/trigger_low()` bracket the loop: the scope arms on the rising edge and `ext_offset` counts cycles from it. The loop counts to 50x50 = 2500 and sends it back. Any landed fault makes the returned count != 2500.

## Baseline: confirm 2500

With no glitch, the target must answer 2500 before we attack it.

In [ ]:
import struct
target.simpleserial_write("g", bytearray([]))
val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)
print(struct.unpack("<I", val['payload'])[0])   # expect 2500

## Set up clock glitching

Classify every shot into three buckets: **normal** (2500), **reset** (no/invalid answer -> reboot), **success** (any other number).

Order matters on the Husky: `cglitch_setup()` reconfigures the MMCM and turns the glitch module **off**, so `scope.glitch.enabled = True` must come **after** it. The print should show `enabled: True`.

In [ ]:
scope.cglitch_setup()                 # clk_src, output=clock_xor, trigger_src=ext_single, io.hs2=glitch
if scope._is_husky:
    scope.glitch.enabled = True       # enable AFTER setup - cglitch_setup turns it off on Husky

# Glitching deliberately crashes the target, which floods the output with
# "Read timed out" / "Device did not ack" warnings. Those are expected, not errors -
# silence them so the sweep/campaign output stays readable (prints still show).
cw.set_all_log_levels(cw.logging.CRITICAL)

print("enabled:", scope.glitch.enabled, "| output:", scope.glitch.output, "| hs2:", scope.io.hs2)

## Glitch controller and live map

`gc` sorts every shot into the three groups and keeps the live counters. The next cell builds the live map: a
bokeh figure wrapped as a `BokehModel` (an ipywidget), which is what lets it stream in VS Code. During the
campaign each shot is streamed onto it - green `+` success, red `x` reset, light-gray dots the swept
background (normal). Click a legend entry to hide that layer (e.g. drop the gray to see faults alone). Run both
cells first, then start the campaign.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
# Live counts only. gc.display_stats() also shows "<param> setting" sliders, but those stay at 0: this
# lab drives scope.glitch directly and only feeds results back with gc.add(), so gc never moves its own
# parameter sliders. The live map below is the real view of what is swept.
gc.widget_list_groups = [widgets.IntText(value=0, description=g + " count:", disabled=True)
                         for g in gc.groups]
display(*gc.widget_list_groups)

In [ ]:
# Live bokeh map - updates per shot in VS Code.
# output_notebook() loads BokehJS (without it the BokehModel widget stays blank);
# BokehModel wraps the figure as an ipywidget, so stream() pushes points live.
from bokeh.io import output_notebook
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
from jupyter_bokeh import BokehModel
from IPython.display import display
output_notebook()

src_normal  = ColumnDataSource(dict(x=[], y=[]))
src_reset   = ColumnDataSource(dict(x=[], y=[]))
src_success = ColumnDataSource(dict(x=[], y=[]))

live_p = figure(width=820, height=600, title="Clock-glitch map",
                x_axis_label="width", y_axis_label="offset")
live_p.scatter("x", "y", source=src_normal,  size=3,  color="lightgray",          legend_label="normal")
live_p.scatter("x", "y", source=src_reset,   size=9,  marker="x",     color="red",   legend_label="reset")
live_p.scatter("x", "y", source=src_success, size=12, marker="cross", color="green", legend_label="success")
live_p.legend.click_policy = "hide"    # click a legend entry to show/hide that layer
display(BokehModel(live_p))

_live_src = {"normal": src_normal, "reset": src_reset, "success": src_success}

def live_add(group, width, offset):
    _live_src[group].stream(dict(x=[width], y=[offset]))

## Campaign - run it for a while and map the band

This is the one experiment cell. It sweeps the whole shuffled grid `PASSES` times (or until `MAX_MINUTES`),
records **every** shot into the three buckets, and feeds each result straight into `gc` - so the live plot
above fills in as it runs. Leave it going to accumulate many successes, resets (crashes) and normals.

Three knobs move together:

- **width / offset** - the shape and position of the extra edge inside one clock period (phase-shift steps,
  `0 .. phase_shift_steps`). `width = 0` is **not** "no glitch": with `clock_xor` a mid-period `offset` alone
  already injects an edge, and that is one of the working points on this target.
- **ext_offset** - which clock cycle after the trigger the glitch lands on. It *must* be swept, not pinned:
  the sensitive cycles here are the odd ones (7, 9, 11...), and a fixed even value like 8 lands in a dead
  phase - no success and not even a reset.
- **repeat = 20** - hammer 20 consecutive cycles per shot. At `repeat = 1` a single-cycle skip is too rare
  to catch; at 20 a fault lands often enough to map a region.

Tune the two knobs at the top: more `PASSES` = denser map, bigger `MAX_MINUTES` = longer run. Each pass over
this grid is ~5-6k shots (a few minutes).

In [ ]:
import struct, random, time

# --- campaign knobs: raise these to run longer / collect more ---
PASSES      = 3     # how many full sweeps of the grid
MAX_MINUTES = 20    # hard wall-clock cap (stops on whichever comes first)
# ----------------------------------------------------------------

# gc.add() keeps the counters/calc; live_add() streams success/reset onto the map.
if scope._is_husky:
    scope.glitch.enabled = True
scope.glitch.repeat = 20
scope.adc.lo_gain_errors_disabled = True
scope.adc.clip_errors_disabled = True
scope.adc.timeout = 0.1

STEPS = scope.glitch.phase_shift_steps
widths  = list(range(0, STEPS // 2, 200))
offsets = list(range(0, STEPS, 200))
exts    = list(range(0, 21))
grid = [(w, o, e) for e in exts for o in offsets for w in widths]

tally = {"success": 0, "reset": 0, "normal": 0}
deadline = time.time() + MAX_MINUTES * 60
t0 = time.time()
n = 0
reboot_flush()
stop = False
for p in range(PASSES):
    random.shuffle(grid)                 # fresh order each pass
    for w, o, e in grid:
        if time.time() > deadline:
            stop = True; break
        n += 1
        if scope.adc.state:              # trigger still high from a crash
            reboot_flush()
        scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset = w, o, e
        scope.arm()
        target.simpleserial_write("g", bytearray([]))
        if scope.capture():              # no trigger -> crashed
            group = "reset"
        else:
            val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)
            if not val['valid'] or val['payload'] is None:
                group = "reset"
            else:
                cnt = struct.unpack("<I", val['payload'])[0]
                group = "normal" if cnt == 2500 else "success"
        tally[group] += 1
        gc.add(group, (w, o, e))
        live_add(group, w, o)
        if group == "reset":
            reboot_flush()
        if n % 500 == 0:
            print(f"{n:6d} shots | {time.time()-t0:4.0f}s | "
                  f"success={tally['success']:4d} reset={tally['reset']:4d} normal={tally['normal']:5d}")
    if stop:
        break

el = time.time() - t0
print(f"\nDONE: {n} shots in {el:.0f}s ({n/max(el,1):.0f}/s), {p+1} pass(es)")
print("tally:", tally, "| success rate: %.2f%%" % (100 * tally['success'] / max(n, 1)))

## Results

The per-setting table, sorted by success rate - use it to read the exact working `width` / `offset` /
`ext_offset`. The live map above already shows the cluster. Once you have a hit, re-sweep a fine step around
it to map the whole working band.

In [ ]:
results = gc.calc(sort="success_rate")
results

## Submission

One screenshot: the working `width` / `offset` / `ext_offset` / `repeat`, a count that is **not 2500**, and the working **range** (the cluster on the map).

> A single hit is luck. A range is a result.

In [ ]:
scope.dis()
target.dis()